In [ ]:
!mkdir -p eval/out

!yosys -q -p "read_verilog eval/fft/fft_n64_w16/*.v; hierarchy -top FFT; proc; opt_merge; opt_clean; pmuxtree; flatten; write_json eval/out/fft_n64_w16.json; synth_xilinx -family xcup; tee -o eval/out/fft_n64_w16.stat stat"

In [ ]:
dsp_rules = {
    "dsp_generic": {
        "requirements": {
            "dsp48e2": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": ["inputs"],
        "outputs": ["outputs"]
    }
}

In [ ]:
import emap
import json

SCHEMA_PATH = "emap/schema.sql"

def simple_cost_model(type_: str, *ports) -> float:
    if type_ == "$dff":
        return len(ports[0]) * 1.0
    elif type_ in {"$muls", "$mulu"}:
        return len(ports[0]) * len(ports[1]) * 1.0
    elif type_ in {"$adds", "$addu", "$subs", "$subu"}:
        return min(len(ports[0]) + len(ports[1]), len(ports[2])) * 1.0
    return len(ports[0]) * 1.0  # other types

TEST_NAME = "fft_n64_w16"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["FFT"], clk="clock")

netlist.rebuild()

complex_mul_matches = emap.rewrites.ematch_complex_mul(netlist)
cnt = emap.rewrites.apply_complex_mul(netlist, complex_mul_matches)
print(f"Applied {cnt} rewrites")
netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$muls"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)
    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

# techmapping
# basis: multiplication
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.techmap_dsp(netlist)

# with open("debug.json", "w") as f:
#     json.dump(netlist.dump_tables(), f, indent=2)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 6}, OutputFlag=False)

with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
!yosys -q -p "read_json eval/out/fft_n64_w16_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/fft_n64_w16_extracted.stat stat"

In [ ]:
!yosys -q -p "read_verilog eval/fft/fft_n128_w16/*.v; hierarchy -top FFT; proc; opt_merge; opt_clean; pmuxtree; flatten; write_json eval/out/fft_n128_w16.json; synth_xilinx -family xcup; tee -o eval/out/fft_n128_w16.stat stat"

In [ ]:
import emap
import json

SCHEMA_PATH = "emap/schema.sql"

def simple_cost_model(type_: str, *ports) -> float:
    if type_ == "$dff":
        return len(ports[0]) * 1.0
    elif type_ in {"$muls", "$mulu"}:
        return len(ports[0]) * len(ports[1]) * 1.0
    elif type_ in {"$adds", "$addu", "$subs", "$subu"}:
        return min(len(ports[0]) + len(ports[1]), len(ports[2])) * 1.0
    return len(ports[0]) * 1.0  # other types

TEST_NAME = "fft_n128_w16"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["FFT"], clk="clock")

netlist.rebuild()

complex_mul_matches = emap.rewrites.ematch_complex_mul(netlist)
cnt = emap.rewrites.apply_complex_mul(netlist, complex_mul_matches)
print(f"Applied {cnt} rewrites")
netlist.rebuild()
# matches = emap.rewrites.ematch_wide_muls(netlist)
# cnt = emap.rewrites.apply_wide_muls_split(netlist, matches)
# print(f"Applied {cnt} rewrites")
# netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$muls"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)
    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

# techmapping
# basis: multiplication
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.techmap_dsp(netlist)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 9}, OutputFlag=False)

with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
!yosys -q -p "read_json eval/out/fft_n128_w16_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/fft_n128_w16_extracted.stat stat"

In [ ]:
!yosys -q -p "read_verilog eval/fft/fft_n1024_w32/*.v; hierarchy -top FFT; proc; opt_merge; opt_clean; pmuxtree; flatten; write_json eval/out/fft_n1024_w32.json; synth_xilinx -family xcup; tee -o eval/out/fft_n1024_w32.stat stat"

In [8]:
import emap
import json

SCHEMA_PATH = "emap/schema.sql"

def simple_cost_model(type_: str, *ports) -> float:
    if type_ == "$dff":
        return len(ports[0]) * 1.0
    elif type_ in {"$muls", "$mulu"}:
        return len(ports[0]) * len(ports[1]) * 1.0
    elif type_ in {"$adds", "$addu", "$subs", "$subu"}:
        return min(len(ports[0]) + len(ports[1]), len(ports[2])) * 1.0
    return len(ports[0]) * 1.0  # other types

TEST_NAME = "fft_n1024_w32"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["FFT"], clk="clock")

netlist.rebuild()

complex_mul_matches = emap.rewrites.ematch_complex_mul(netlist)
cnt = emap.rewrites.apply_complex_mul(netlist, complex_mul_matches)
print(f"Applied {cnt} rewrites")
netlist.rebuild()
matches = emap.rewrites.ematch_wide_muls(netlist)
cnt = emap.rewrites.apply_wide_muls_split(netlist, matches)
print(f"Applied {cnt} rewrites")
netlist.rebuild()

cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$muls"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)
    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

# techmapping
# basis: multiplication
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.techmap_dsp(netlist)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 36}, OutputFlag=False)

with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

Found 18868 cells
Processing cell 0/18868: $flatten\SU1.$add$eval/fft/fft_n1024_w32/sdfunit.v:107$310
Processing cell 1000/18868: $flatten\SU1.\DB1.$procdff$3518
Processing cell 2000/18868: $flatten\SU1.\TW.$auto$pmuxtree.cc:35:or_generator$8411
Processing cell 3000/18868: $flatten\SU1.\TW.$auto$pmuxtree.cc:65:recursive_mux_generator$4935
Processing cell 4000/18868: $flatten\SU1.\TW.$auto$pmuxtree.cc:65:recursive_mux_generator$7933
Processing cell 5000/18868: $flatten\SU1.\TW.$procmux$1873_CMP0
Processing cell 6000/18868: $flatten\SU2.\DB1.$procdff$4340
Processing cell 7000/18868: $flatten\SU2.\TW.$auto$pmuxtree.cc:37:or_generator$7491
Processing cell 8000/18868: $flatten\SU2.\TW.$auto$pmuxtree.cc:65:recursive_mux_generator$6241
Processing cell 9000/18868: $flatten\SU2.\TW.$auto$pmuxtree.cc:65:recursive_mux_generator$9239
Processing cell 10000/18868: $flatten\SU2.\TW.$procmux$2309_CMP0
Processing cell 11000/18868: $flatten\SU3.\TW.$auto$pmuxtree.cc:37:or_generator$10315
Processing cell

In [9]:
!yosys -q -p "read_json eval/out/fft_n1024_w32_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/fft_n1024_w32_extracted.stat stat"